## From Parameters to Weights

A Threedx model is defined by its three parameters: `alpha`, `alpha_seasonal`, and `alpha_seasonal_decay`. Together, the three parameters define the weights that a Threedx model assigns to past observations when calculating a forecast. To understand the meaning of the parameters and their impact on a Threedx forecast, you need to understand how the three parameters are projected into weights space.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import threedx as tdx

### Understanding the Individual Parameters

The best way to understand how the three parameters interact is to first understand how they work individually. Each of the parameters can be used to derive weights; a Threedx model in the end only combines the three individual weight vectors into a combined one.

#### Common Characteristics

Before looking at each parameter individually, keep in mind that the following characteristics hold for each of the three parameters:

- Each parameter can take values in the range of [0, 1]
- Threedx weight vectors always sum up to 1

#### Understanding `alpha`

The `alpha` parameter creates exponential weights just like the `alpha` parameter in [Simple Exponential Smoothing](https://en.wikipedia.org/wiki/Exponential_smoothing#Basic_(simple)_exponential_smoothing), following the update equation:

```python
w_t = alpha * ((1-alpha) ** (t-1))
```

You can generate a vector of `n` exponential smoothing weights using `alpha` as follows:

In [ ]:
tdx.weights_exponential(alpha=0.1, n=5)

To align with time series, Threedx weight vectors always range from the weight for the oldest observation on the left, to the weight for the most recent observation on the right. Exponential weights specifically assign the largest weight to the most recent observation and decrease monotonically from there to the smallest weight being assigned to the oldest observation.

Note that Threedx weight vectors are always standardized to sum to one. This has a small impact if `alpha` or `n` are large. In the example above, however, the most recent weight is 0.24 instead of 0.1 because the smoothing factor `alpha` is small (implying little smoothing) while `n`, too, is small.

Non-normalized weights would look as follows (note that the most recent weight is equal to `alpha`):

In [ ]:
n = 5
alpha = 0.1

weights_non_normalized = alpha * ((1 - alpha) ** np.arange(n)[::-1])

print(f"{weights_non_normalized=}")
print(f"{weights_non_normalized.sum()=}")

With sufficiently large `n`, normalization is barely needed as the sum of exponential weights approaches one:

In [ ]:
n = 100
weights_non_normalized_large_n = alpha * ((1 - alpha) ** np.arange(n)[::-1])

print(f"{weights_non_normalized_large_n.sum()=}")

At the one extreme of `alpha=0`, weights are distributed uniformly and would result in a forecast equal to the simple average of past observations:

In [ ]:
tdx.weights_exponential(alpha=0., n=5)

At the other extreme of `alpha=1`, all weight is assigned to the most recent observation and would thus result in a naive forecast:

In [ ]:
tdx.weights_exponential(alpha=1., n=5)

All other `alpha` values interpolate smoothly between the two extremes:

In [ ]:
print(f"alpha=0.00: {tdx.weights_exponential(alpha=0.00, n=5)}")
print(f"alpha=0.01: {tdx.weights_exponential(alpha=0.01, n=5)}")
print(f"alpha=0.10: {tdx.weights_exponential(alpha=0.10, n=5)}")
print(f"alpha=0.50: {tdx.weights_exponential(alpha=0.50, n=5)}")
print(f"alpha=0.90: {tdx.weights_exponential(alpha=0.90, n=5)}")
print(f"alpha=0.99: {tdx.weights_exponential(alpha=0.99, n=5)}")
print(f"alpha=1.00: {tdx.weights_exponential(alpha=1.00, n=5)}")

#### Understanding `alpha_seasonal_decay`

While normally the *third* parameter, it makes sense to continue with `alpha_seasonal_decay` for the purpose of this tutorial. Both `alpha_seasonal` and `alpha_seasonal_decay` extend the idea of exponential smoothing weights to seasonal time series. To generate weight vectors with either of the two parameters, it's necessary to supply an additional `period_length` argument to define the expected period length of the time series' seasonal component.

Consequently, a weight vector based on `alpha_seasonal_decay` can be generated as follows (here for a weekly period length as could be used for daily observations):

In [ ]:
tdx.weights_seasonal_decay(alpha=0.1, n=35, period_length=7)

The seasonal decay weights steered by `alpha_seasonal_decay` only change *outside* of seasonal periods or *across* seasonal periods, but are constant *within* seasonal periods. Starting from the most recent observation (the last weight in the vector), weights are constant for `period_length` weights, then change to a new value for the next `period_length` weights, and so on.

Across periods, weights decrease again exponentially. This fact is a bit hidden in the above example due to the standardization of the weight vector to sum to one. It can be recovered by considering the unique values and re-normalizing them. They then match the exponential weights where `alpha=alpha_seasonal_decay`.

In [ ]:
weights_seasonal_decay_unique = np.unique(
    tdx.weights_seasonal_decay(alpha=0.1, n=35, period_length=7)
)

weights_seasonal_decay_unique_normalized = \
    weights_seasonal_decay_unique / weights_seasonal_decay_unique.sum()

print(f"{weights_seasonal_decay_unique=}")
print(f"{weights_seasonal_decay_unique_normalized=}")
print(f"{tdx.weights_exponential(alpha=0.1, n=5)=}")

Or set `period_length=1` to arrive at the same conclusion:

In [ ]:
print(f"{tdx.weights_seasonal_decay(alpha=0.1, n=5, period_length=1)=}")

The fact that weights only change *across* periods results in a step function pattern that becomes even more obvious when plotting the weight vectors:

In [ ]:
plt.bar(
    x=np.arange(50),
    height=tdx.weights_seasonal_decay(alpha=0.1, n=50, period_length=7),
)

At the extreme of `alpha_seasonal_decay=0.`, weights are perfectly uniform:

In [ ]:
plt.bar(
    x=np.arange(50),
    height=tdx.weights_seasonal_decay(alpha=0.0, n=50, period_length=7),
)

At the other extreme of `alpha_seasonal_decay=1.0`, all weight is assigned to the most recent period (and within that period uniformly). The resulting weight vector would generate a latest-period-average forecast.

In [ ]:
plt.bar(
    x=np.arange(50),
    height=tdx.weights_seasonal_decay(alpha=1.0, n=50, period_length=7),
)

#### Understanding `alpha_seasonal`

Like `alpha_seasonal_decay`, `alpha_seasonal` transforms into weights that account for a seasonal component in time series. While `alpha_seasonal_decay` transforms into weights that vary *across* periods, `alpha_seasonal` transforms into weights that vary *within* a seasonal period but stay constant across periods.

This is understood easiest by looking at the resulting weight vector:

In [ ]:
plt.bar(
    x=np.arange(50),
    height=tdx.weights_seasonal(alpha=0.1, n=50, period_length=7),
)

Within a given period, `alpha_seasonal` distributes weights by assigning the most weight to the observation at the beginning of the period and the least weight to observations in the middle of a period. At the extreme of `alpha_seasonal=1`, this results in a seasonal average forecast. At values between zero and one, this results in forecasts primarily based on past observations that are similar to the predicted point in time in terms of their position within the seasonal period.

In [ ]:
print(f"alpha=0.00: {tdx.weights_seasonal(alpha=0.00, n=5, period_length=5)}")
print(f"alpha=0.01: {tdx.weights_seasonal(alpha=0.01, n=5, period_length=5)}")
print(f"alpha=0.10: {tdx.weights_seasonal(alpha=0.10, n=5, period_length=5)}")
print(f"alpha=0.50: {tdx.weights_seasonal(alpha=0.50, n=5, period_length=5)}")
print(f"alpha=0.90: {tdx.weights_seasonal(alpha=0.90, n=5, period_length=5)}")
print(f"alpha=0.99: {tdx.weights_seasonal(alpha=0.99, n=5, period_length=5)}")
print(f"alpha=1.00: {tdx.weights_seasonal(alpha=1.00, n=5, period_length=5)}")

### Understanding the Combined Parameters

The final weight vector used by a Threedx model is a combination of the three individual parameters' weight vectors. Consequently, it will exhibit a combination of the three individual weight vectors' characteristics. This is again most transparent when plotting the weights:

In [ ]:
weights_threedx = tdx.weights_threedx(
    alpha=0.01,
    alpha_seasonal=0.1,
    alpha_seasonal_decay=0.1,
    n=50,
    period_length=7
)

plt.bar(x=np.arange(50), height=weights_threedx)

The combined weight vector is simply the product of the three vectors, normalized to sum up to one. The weights generated by `tdx.weights_threedx()` can be recreated using the three individual weight functions as follows:

In [ ]:
combined_weights = \
    tdx.weights_exponential(alpha=0.01, n=50) * \
    tdx.weights_seasonal(alpha=0.1, n=50, period_length=7) * \
    tdx.weights_seasonal_decay(alpha=0.1, n=50, period_length=7)

combined_weights = combined_weights / combined_weights.sum()

print(f"{combined_weights=}")
print(f"{weights_threedx=}")

### Understanding the Edge Cases

With its three parameters, a Threedx model can fit a wide range of different weights that interpolate between a set of edge cases.

#### Simple Average Forecast

By setting each parameter to zero, a Threedx model assigns weights to past observations that result in a simple average forecast. The weights are perfectly uniform.

In [ ]:
weights_simple_average = tdx.weights_threedx(
    alpha=0.,
    alpha_seasonal=0.,
    alpha_seasonal_decay=0.,
    n=50,
    period_length=7
)

plt.bar(x=np.arange(50), height=weights_simple_average)

#### Naive Forecast

Whenever the `alpha` parameter is set to one, independent of the other two parameters, all weight will be assigned to the most recent observation. This results in a naive forecast.

In [ ]:
weights_naive = tdx.weights_threedx(
    alpha=1.,
    alpha_seasonal=0.,
    alpha_seasonal_decay=0.,
    n=50,
    period_length=7
)

plt.bar(x=np.arange(50), height=weights_naive)

#### Seasonal Average Forecast

A seasonal average forecast requires uniform weights on the observations *k*-periods ago, and zero weight on observations in between. This can be achieved by choosing `alpha=0` and `alpha_seasonal_decay=0` (avoiding decreasing weights) and choosing `alpha_seasonal=1` to assign all weight within a period onto a single observation in the period.

In [ ]:
weights_seasonal_average = tdx.weights_threedx(
    alpha=0.,
    alpha_seasonal=1.,
    alpha_seasonal_decay=0.,
    n=50,
    period_length=7
)

plt.bar(x=np.arange(50), height=weights_seasonal_average)

#### Seasonal Naive Forecast

In contrast to the seasonal average, the seasonal naive forecast uses only the most recent period (and within that period the observation a period ago). This means you want extreme decay across periods and thus change to `alpha_seasonal_decay=1`:

In [ ]:
weights_seasonal_naive = tdx.weights_threedx(
    alpha=0.,
    alpha_seasonal=1.,
    alpha_seasonal_decay=1.,
    n=50,
    period_length=7
)

plt.bar(x=np.arange(50), height=weights_seasonal_naive)

#### Latest Period Average Forecast

Finally, switch to `alpha_seasonal=0` to keep assigning all weight to the most recent period. This will, however, distribute the weight uniformly across all observations within that period. The resulting forecast is the average of the observations in the most recent period.

In [ ]:
weights_latest_period_average = tdx.weights_threedx(
    alpha=0.,
    alpha_seasonal=0.,
    alpha_seasonal_decay=1.,
    n=50,
    period_length=7
)

plt.bar(x=np.arange(50), height=weights_latest_period_average)

### Choosing the Parameter Grid

You need a solid understanding of how parameters are projected into weight space, and how weights result into forecasts when you define the parameter grid over which a Threedx model is fitted. As shown in the [Get Started](https://timradtke.github.io/threedx/howto/getstarted.html) section, you need to provide a parameter grid when you initialize the [Threedx model object](https://timradtke.github.io/threedx/api/threedx/threedx.html#threedx.Threedx).

The [threedx.initialize](https://timradtke.github.io/threedx/api/threedx/threedx.initialize.html) module provides functions that make it easy to define such a grid.

For example, you know understand how to interpret the parameter combinations that the [threedx.initialize.initialize_edge_case_parameters()](https://timradtke.github.io/threedx/api/threedx/threedx.initialize.html#threedx.initialize.initialize_edge_case_parameters) function generates:

In [ ]:
tdx.initialize_edge_case_parameters()

The edge cases are included per default also when you create a larger set of parameter combinations using [initialize_parameters_at_random()](https://timradtke.github.io/threedx/api/threedx/threedx.html#threedx.initialize_parameters_at_random) or [initialize_parameters_in_grid()](https://timradtke.github.io/threedx/api/threedx/threedx.html#threedx.initialize_parameters_in_grid):

In [ ]:
tdx.initialize_parameters_at_random(size=10, include_edge_cases=True)

But all of these functions are just provided to get you started. You can define the parameter grid in any way you like. Apply the understanding you gained throughout this tutorial to fit a Threedx model over parameters that make sense for your use case.

For example, you could restrict the range of models that Threedx will fit to Simple Exponential Smoothing models by only varying the `alpha` parameter and setting `alpha_seasonal` and `alpha_seasonal_decay` to zero, as follows:

In [ ]:
grid_simple_exponential_smoothing = np.zeros((10, 3))
grid_simple_exponential_smoothing[:, 0] = np.arange(0, 1, 0.1)
print(grid_simple_exponential_smoothing)